# Train LoRA checkpoints on Colab
Select a GPU runtime and run all cells. The defaults train nine FP16 adapters for the local full sweep, one for each combination of three seeds and three training sizes. Training runs on the VM. Each completed checkpoint is published to Google Drive. The notebook uses MBPP and the base model; it does no benchmark generation or scoring.

For the Mac CLI run, the pinned checkout’s `configs/full.toml` supplies the exact target configuration. To target a run with different settings (including a run from **Full_Sweep.ipynb**), set `TARGET_MANIFEST` to its saved `manifest.json` and use its code revision. The adapter identity includes the **entire** config, selected examples, validation rows, and seed. Filtering jobs below leaves that identity intact. Matching seeds reproduce example selection and the dialect; floating-point training can differ across GPU backends.


In [ ]:
REVISION = "38daf7cd68699438c0a5b6f6400877572f4c0137"
TARGET_MANIFEST = None  # Example: "/content/drive/MyDrive/cs182-sweep/runs/<name>/manifest.json"
SEEDS = None  # None selects all configured seeds; e.g. [0] for one shard
SIZES = None  # None selects all configured sizes; e.g. [16, 64]

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/cs182-sweep")
WORK = Path("/content/cs182-lora-training")
WORK.mkdir(parents=True, exist_ok=True)
repo_url = "https://github.com/KG2468/ICLvsLoRA.git"
repo = WORK / "repo"
if not repo.exists():
    source = DRIVE / "repo"
    subprocess.run(["git", "clone", "--no-checkout", str(source) if (source / ".git").exists() else repo_url, str(repo)], check=True)
if subprocess.run(["git", "-C", str(repo), "cat-file", "-e", REVISION + "^{commit}"], capture_output=True).returncode:
    subprocess.run(["git", "-C", str(repo), "fetch", repo_url, REVISION], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REVISION], check=True)
manifest = json.loads(Path(TARGET_MANIFEST).read_text()) if TARGET_MANIFEST else None
cfg = manifest["config"] if manifest else tomllib.loads((repo / "configs/full.toml").read_text())
exp = cfg["experiment"]
def select(values, requested):
    if requested is None:
        return values
    if not requested or not set(requested).issubset(values):
        raise ValueError(f"Select a nonempty subset of {values}")
    return [value for value in values if value in requested]

jobs = [(seed, n) for seed in select(exp["seeds"], SEEDS) for n in select(exp["sizes"], SIZES)]
model_dir = Path(cfg["model"]["directory"]) / cfg["model"]["precision"]
data_dir = Path(exp["data_dir"])
run_dir = Path(exp["output_dir"]) / exp["name"]
for path in (data_dir, model_dir, run_dir):
    if path.is_absolute() or ".." in path.parts:
        raise ValueError(f"Target config requires project-relative paths: {path}")
destination = DRIVE / run_dir / "adapters"
print("Jobs (seed, N):", jobs)
print("Drive checkpoint directory:", destination)


## Set up the training environment


In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Select a Colab NVIDIA GPU runtime and rerun.")
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap",
                               "--format=csv,noheader"], text=True).splitlines()[0]
name, version, capability = [part.strip() for part in gpu.split(",")]
if float(capability) < 7.5 or tuple(map(int, version.split("."))) < (550, 54, 14):
    raise RuntimeError(f"Unsupported GPU or driver: {gpu}")
mlx = "mlx[cuda13]==0.32.4" if int(version.split(".")[0]) >= 580 else "mlx[cuda12]==0.32.4"
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
uv = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
venv = WORK / "venv"
subprocess.run([uv, "python", "install", "3.12.11"], check=True)
py = venv / "bin/python"
if not py.exists():
    subprocess.run([uv, "venv", "--python", "3.12.11", str(venv)], check=True)
subprocess.run([uv, "pip", "install", "--python", str(py), mlx, "mlx-lm[train]==0.32.2",
                "datasets==4.8.5", "transformers==5.23.0", "huggingface-hub==1.33.0"], check=True)
env = {**os.environ, "PYTHONPATH": str(repo / "src"), "PYTHONUNBUFFERED": "1"}
def run(*args, cwd=WORK):
    subprocess.run([str(py), *args], env=env, cwd=cwd, check=True)
run("-c", "import mlx.core as mx; mx.set_default_device(mx.gpu); print(mx.device_info(), mx.ones(1).item())")


## Stage the base model and MBPP locally
Cached inputs are copied from Drive when available. Missing inputs are downloaded on the VM. When a target manifest is supplied, its dataset and weight hashes are checked before training.


In [ ]:
settings = WORK / "target-config.json"
settings.write_text(json.dumps(cfg))
local_data = WORK / data_dir
local_data.mkdir(parents=True, exist_ok=True)
for name in ("mbpp_train.jsonl", "mbpp_valid.jsonl", "manifest.json"):
    cached = DRIVE / data_dir / name
    if cached.is_file() and not (local_data / name).exists():
        shutil.copy2(cached, local_data / name)
load_cfg = "import json, sys; cfg = json.load(open(sys.argv[1])); "
run("-c", load_cfg + "from final_project.data import prepare; cfg['experiment']['benchmarks'] = []; prepare(cfg)", str(settings))
local_model = WORK / model_dir
if not local_model.exists():
    local_model.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=WORK) as tmp:
        staged = Path(tmp) / model_dir
        cached = DRIVE / model_dir
        if cached.is_dir():
            shutil.copytree(cached, staged)
        run("-c", load_cfg + "from final_project.backend import convert; convert(cfg)", str(settings), cwd=tmp)
        staged.rename(local_model)
else:
    run("-c", load_cfg + "from final_project.backend import verify_model; verify_model(cfg)", str(settings))

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

if manifest:
    for key in ("mbpp_train", "mbpp_valid"):
        assert sha256(local_data / (key + ".jsonl")) == manifest["datasets"][key], f"Dataset mismatch: {key}"
    for name, expected in manifest["weights"].items():
        assert sha256(local_model / name) == expected, f"Model mismatch: {name}"


## Train, verify, and publish each checkpoint
Each job calls the sweep’s training entry point in a fresh process. Rerunning verifies completed checkpoints and skips their training. A disconnect during a job requires repeating that job. When splitting work across training VMs, choose disjoint seed/size pairs.


In [ ]:
files = ("adapter_config.json", "adapters.safetensors", "complete.json")

def verify(directory):
    json.loads((directory / "adapter_config.json").read_text())
    marker = json.loads((directory / "complete.json").read_text())
    if sha256(directory / "adapters.safetensors") != marker["weights_sha256"]:
        raise ValueError(f"Weight checksum mismatch: {directory}")
    return marker

destination.mkdir(parents=True, exist_ok=True)
for seed, n in jobs:
    label = f"seed{seed}-n{n}"
    saved = destination / label
    with tempfile.TemporaryDirectory(dir=WORK) as tmp:
        adapter = Path(tmp) / label
        if saved.exists():
            adapter.mkdir()
            for filename in files:
                shutil.copy2(saved / filename, adapter / filename)
            verify(adapter)
        job = Path(tmp) / "job.json"
        job.write_text(json.dumps({"cfg": cfg, "seed": seed, "n": n, "adapter": str(adapter)}))
        # The trainer also checks the exact target config and selected rows on reuse.
        run("-m", "final_project", "_train", str(job))
        marker = verify(adapter)
        if not saved.exists():
            with tempfile.TemporaryDirectory(dir=destination, prefix=".staging-") as scratch:
                staged = Path(scratch) / label
                staged.mkdir()
                for filename in files:
                    shutil.copy2(adapter / filename, staged / filename)
                verify(staged)
                staged.rename(saved)
        print(label, marker["iterations"], "updates;", saved)
print(f"{len(jobs)} requested checkpoints verified.")


## Use the adapters in the evaluation run
For a Mac run, copy the `seed…-n…` folders from the printed Drive directory into `<project>/runs/<target experiment name>/adapters/` before its LoRA stage starts. For a fresh Colab full sweep using the same target manifest, the folders are already in the Drive restore layout. If that evaluator is already running, copy newly completed folders into its local run directory before its LoRA stage. The evaluator validates the saved identity and weight checksum when it reuses each adapter.
